# Fahrrad-Sharing: Nachfrage vorhersagen mit Maschinellem Lernen

In diesem Notebook trainieren Sie Schritt für Schritt ein Vorhersagemodell auf Basis tabellarischer Daten. Ziel ist es, die **Nachfrage nach Leihfahrrädern an einem Tag** vorherzusagen, und zwar anhand von Informationen wie Wochentag, Jahreszeit und Wetter.

Die Zielgröße ist eine Anzahl, also eine kontinuierliche Zahl. Deshalb sprechen wir von einem **Regressionsproblem**. Genau dasselbe Vorgehen funktioniert in der Produktion, etwa wenn man aus Prozessparametern eine Qualitätskenngröße oder den Energieverbrauch vorhersagen möchte.

### So funktioniert dieses Notebook
- Ein Notebook besteht aus **Textzellen** (wie dieser) und **Code-Zellen** (grau hinterlegt).
- Eine Code-Zelle führen Sie aus, indem Sie hineinklicken und **Shift + Enter** drücken. Das Ergebnis erscheint direkt darunter.
- Führen Sie die Zellen **der Reihe nach von oben nach unten** aus. Spätere Zellen bauen auf früheren auf.
- An manchen Stellen steht im Code **`XXXX`**. Das sind Lücken, die Sie selbst ausfüllen. Die Hinweise im Text darüber helfen Ihnen dabei.
- Python läuft hier komplett **in Ihrem Browser**. Ihre Änderungen werden automatisch in diesem Browser gespeichert. Wenn Sie das Notebook behalten möchten, laden Sie es am Ende herunter (Rechtsklick auf die Datei im Dateibrowser links → *Download*).
- Wenn etwas hakt: Im Menü *Kernel → Restart Kernel and Run All Cells…* startet alles neu.

## Inhaltsverzeichnis
1. Benötigte Bibliotheken importieren
2. Daten laden und ansehen
3. Merkmale und Zielgröße auswählen
4. Daten bereinigen
5. Daten in Trainings- und Testdaten aufteilen
6. Erstes Modell trainieren: Lineare Regression
7. Modell testen
8. Modell bewerten
9. Ein stärkeres Modell: Random Forest
10. Neuronales Netz
11. Alle Modelle im Vergleich
12. Modell speichern und laden
13. Zum Weiterprobieren

## 1. Benötigte Bibliotheken importieren

Wir nutzen drei bewährte Python-Bibliotheken:
- **`pandas`** für Tabellen (sogenannte *DataFrames*),
- **`numpy`** für schnelle Berechnungen mit Zahlen,
- **`matplotlib`** für Diagramme.

Die eigentlichen Lernverfahren kommen später aus der Bibliothek **`scikit-learn`** (kurz `sklearn`). Sie bietet viele Verfahren für Regression, Klassifikation und Clustering mit einer einheitlichen Bedienung.

Beim ersten Ausführen dauert die Zelle einige Sekunden, weil der Browser die Bibliotheken nachlädt.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Einheitliche Größe für alle Diagramme
plt.rcParams["figure.figsize"] = (14, 4.5)
print("Bibliotheken geladen.")

## 2. Daten laden und ansehen

Der Datensatz enthält für jeden Tag der Jahre 2011 und 2012 die Anzahl ausgeliehener Fahrräder eines Bike-Sharing-Anbieters in Washington D.C., zusammen mit Kalender- und Wetterinformationen. Hintergrund zum Datensatz: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset).

Wir laden die Datei `bike_sharing.csv` mit der Funktion `read_csv()` aus `pandas` und speichern die Tabelle in der Variablen `df` (kurz für *DataFrame*). Die Datumsspalte `dteday` lassen wir direkt als Datum einlesen, damit die Diagramme später eine saubere Zeitachse haben.

In [ ]:
try:
    df = pd.read_csv("bike_sharing.csv", parse_dates=["dteday"])
except FileNotFoundError:
    # Sicherheitsnetz: Falls der Browser den Dateizugriff blockiert,
    # wird die Datei direkt von der Webseite geladen.
    import js
    from pyodide.http import open_url
    url = str(js.location.href).split("/extensions/")[0] + "/files/bike_sharing.csv"
    df = pd.read_csv(open_url(url), parse_dates=["dteday"])

print("Anzahl Zeilen (Tage):", len(df))
print("Anzahl Spalten:", len(df.columns))

Mit der Methode `head()` sehen wir uns die ersten fünf Zeilen an.

**Lücke:** Ergänzen Sie den Methodennamen.

In [ ]:
df.XXXX()

### Was steht in den Spalten?

Man unterscheidet zwei Arten von Spalten:
- **Merkmale** (engl. *features*) sind Einflussgrößen, aus denen das Modell lernt, z. B. Wetter oder Wochentag.
- **Zielgrößen** (engl. *labels*) sind die Werte, die wir vorhersagen möchten.

**Merkmale**
- **dteday**: Datum
- **season**: Jahreszeit (als Text: `spring`, `summer`, `fall`, `winter`)
- **yr**: Jahr (0 = 2011, 1 = 2012)
- **mnth**: Monat (1 bis 12)
- **holiday**: Feiertag (0 = nein, 1 = ja)
- **weekday**: Wochentag (als Text: `monday` … `sunday`)
- **workingday**: Arbeitstag (1 = weder Wochenende noch Feiertag, sonst 0)
- **weathersit**: Wetterlage (1 = klar bis leicht bewölkt, 2 = bewölkt/Nebel, 3 = leichter Regen/Schnee, 4 = starker Regen/Schnee)
- **temp**: Temperatur (normiert auf 0 bis 1)
- **atemp**: gefühlte Temperatur (normiert)
- **hum**: Luftfeuchtigkeit (normiert)
- **windspeed**: Windgeschwindigkeit (normiert)

**Zielgrößen**
- **casual**: Anzahl Ausleihen durch nicht registrierte Gelegenheitsnutzer
- **registered**: Anzahl Ausleihen durch registrierte Nutzer
- **cnt**: Gesamtzahl der Ausleihen (`casual` + `registered`)

Es gibt also **drei mögliche Zielgrößen**. Für jede würde man ein eigenes Modell trainieren. Außerdem gibt es **Text-Merkmale** (`season`, `weekday`) und **Zahlen-Merkmale**. Das wird später noch wichtig.

Wie in echten Produktionsdaten ist der Datensatz nicht perfekt: Es gibt **fehlende Werte** und **fehlerhafte Datenpunkte**. Einen ersten Überblick liefert `describe()`. Die Methode zeigt für jede Zahlenspalte unter anderem Minimum, Maximum und Mittelwert.

In [ ]:
df.describe().round(2)

Schauen Sie sich die Zeilen `min` und `max` der Spalte `casual` an. Kommen Ihnen die Werte plausibel vor?

Mit `isna().sum()` zählen wir zusätzlich, wie viele Werte pro Spalte fehlen (*NaN* = „Not a Number“).

In [ ]:
df.isna().sum()

## 3. Merkmale und Zielgröße auswählen

Damit `scikit-learn` ein Modell trainieren kann, müssen wir festlegen, welche Spalten Merkmale sind und welche die Zielgröße ist.

Wir wählen zunächst **`casual`** als Zielgröße, wollen also die Anzahl der Gelegenheitsnutzer pro Tag vorhersagen. Das Datum selbst lassen wir als Merkmal weg, da die Informationen daraus bereits in `yr`, `mnth` und `weekday` stecken.

**Lücken:** Ergänzen Sie das fehlende Merkmal (die Windgeschwindigkeit) und die Zielgröße. Achten Sie auf die genaue Schreibweise der Spaltennamen in Anführungszeichen.

In [ ]:
features = ["season", "yr", "mnth", "holiday", "weekday", "workingday", "weathersit",
            "temp", "atemp", "hum", "XXXX"]
label = "XXXX"

Um ein Gefühl für die Daten zu bekommen, stellen wir die Zielgröße über die Zeit dar.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["dteday"], df[label])
ax.set_ylabel("Ausleihen pro Tag")
ax.set_title(f"Zielgröße '{label}' über die Zeit")
ax.grid()
plt.show()

Im Diagramm fallen einige **Ausreißer** auf: einzelne Tage mit extrem hohen Werten und sogar negative Werte, die es bei einer Anzahl gar nicht geben kann. Solche Fehler entstehen zum Beispiel durch Tippfehler oder Softwarefehler in einer Datenbank. In der Produktion sind es typischerweise Messfehler oder defekte Sensoren.

## 4. Daten bereinigen

Ausreißer und fehlende Werte stören das Training und können das Ergebnis deutlich verfälschen. Man entfernt sie daher, oder man ersetzt sie durch plausible Werte, etwa durch Interpolation zwischen den Nachbarwerten.

Die einfachste Methode ist die **Sichtprüfung**: Wir lesen im Diagramm oben ab, in welchem Bereich gültige Werte liegen, und legen eine Ober- und eine Untergrenze fest. Alle Zeilen außerhalb dieser Grenzen werden gelöscht.

**Lücken:**
- Wählen Sie anhand des Diagramms eine sinnvolle **Obergrenze**. Die normalen Spitzenwerte sollen erhalten bleiben, die extremen Ausreißer nicht.
- Wählen Sie eine **Untergrenze**, die die unmöglichen negativen Werte entfernt.
- Tragen Sie bei beiden Filtern den Namen der Spalte ein, die bereinigt werden soll.

In [ ]:
# Grenzen per Sichtprüfung aus dem Diagramm oben
obergrenze = XXXX
untergrenze = XXXX

# Nur Zeilen behalten, deren Wert unterhalb der Obergrenze liegt
df = df[df["XXXX"] < obergrenze]

# Nur Zeilen behalten, deren Wert oberhalb der Untergrenze liegt
df = df[df["XXXX"] > untergrenze]

# Zeilen mit fehlenden Werten (NaN) entfernen
df = df.dropna()

# Zeilennummern neu durchzählen
df = df.reset_index(drop=True)
print("Verbleibende Tage:", len(df))

Nun prüfen wir das Ergebnis erneut im Diagramm.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["dteday"], df[label])
ax.set_ylabel("Ausleihen pro Tag")
ax.set_title(f"Zielgröße '{label}' nach der Bereinigung")
ax.grid()
plt.show()

Jetzt erkennt man die eigentlichen Muster:
- Einen deutlichen **saisonalen Effekt**: Im Sommer wird viel mehr geliehen als im Winter.
- Ein **Wachstum von 2011 auf 2012**.
- Starke Schwankungen **innerhalb jeder Woche**: Gelegenheitsnutzer fahren vor allem am Wochenende.

Ein gutes Modell sollte genau diese Muster aus den Merkmalen lernen.

## 5. Daten in Trainings- und Testdaten aufteilen

Ein Modell soll später Tage vorhersagen, die es noch nie gesehen hat. Um das ehrlich zu prüfen, halten wir einen Teil der Daten zurück:
- **Trainingsdaten** (die ersten 80 % der Tage): Daraus lernt das Modell.
- **Testdaten** (die letzten 20 % der Tage): Damit prüfen wir am Ende, wie gut das Modell auf unbekannten Daten ist.

Wir teilen hier **zeitlich** auf, also Vergangenheit zum Lernen und Zukunft zum Testen. Das entspricht dem echten Einsatz, denn auch in der Praxis soll ein Modell die Zukunft vorhersagen.

Die Methode `iloc[...]` wählt Zeilen nach ihrer Position aus. `iloc[:train_size]` bedeutet „alle Zeilen bis zur Position `train_size`“, `iloc[train_size:]` „alle Zeilen ab dieser Position“.

**Lücken:** Wir benötigen jeweils die Merkmale (`X`) und die Zielgröße (`y`) für Training und Test. Setzen Sie die passenden Variablen aus Abschnitt 3 ein.

In [ ]:
# Anzahl der Zeilen für die Trainingsdaten
train_size = round(0.8 * len(df))

X_train = df.iloc[:train_size][XXXX]
X_test  = df.iloc[train_size:][XXXX]
y_train = df.iloc[:train_size][XXXX]
y_test  = df.iloc[train_size:][XXXX]

print("Trainingstage:", len(X_train), "| Testtage:", len(X_test))

## 6. Erstes Modell trainieren: Lineare Regression

Wir beginnen mit dem einfachsten Verfahren, der **linearen Regression**. Sie stellt die Zielgröße als gewichtete Summe der Merkmale dar und lernt die passenden Gewichte aus den Trainingsdaten.

In `scikit-learn` läuft jedes Verfahren gleich ab:
1. Ein Modell-Objekt erzeugen, hier `LinearRegression()`.
2. Mit der Methode **`fit()`** trainieren. Dabei übergeben wir die Merkmale `X_train` und die Zielgröße `y_train`.
3. Mit der Methode **`predict()`** Vorhersagen machen.

Probieren wir es aus:

In [ ]:
from sklearn.linear_model import LinearRegression

regression = LinearRegression()
regression.fit(X_train, y_train)

Die Zelle bricht mit einem Fehler ab, der ungefähr so lautet: `ValueError: could not convert string to float: 'winter'`.

Das ist ein typischer und lehrreicher Fehler. Lernverfahren rechnen ausschließlich mit **Zahlen**. Die Spalten `season` und `weekday` enthalten aber Text. Wir müssen sie also zuerst in Zahlen übersetzen. Dafür legen wir je eine Zuordnungstabelle an (in Python ein *Dictionary*) und wenden sie mit der Methode `map()` an.

**Lücken:** Vergeben Sie für die Jahreszeiten die Zahlen 0 bis 3 und für die Wochentage die Zahlen 0 bis 6, jeweils in der natürlichen Reihenfolge.

In [ ]:
jahreszeiten = {"spring": XXXX, "summer": XXXX, "fall": XXXX, "winter": XXXX}
wochentage = {"monday": XXXX, "tuesday": XXXX, "wednesday": XXXX, "thursday": XXXX,
              "friday": XXXX, "saturday": XXXX, "sunday": XXXX}

df["season"] = df["season"].map(jahreszeiten)
df["weekday"] = df["weekday"].map(wochentage)

# Aufteilung wiederholen, damit die neuen Zahlenspalten übernommen werden
X_train = df.iloc[:train_size][features]
X_test  = df.iloc[train_size:][features]
y_train = df.iloc[:train_size][label]
y_test  = df.iloc[train_size:][label]

X_train.head()

Alle Merkmale sind jetzt Zahlen. Wir trainieren das Modell erneut.

In [ ]:
regression = LinearRegression()
regression.fit(X_train, y_train)
print("Training abgeschlossen.")

Das Modell hat ohne Fehler gelernt. Bei der linearen Regression können wir sogar direkt nachsehen, **welches Gewicht** jedes Merkmal bekommen hat. Ein positives Gewicht bedeutet: Steigt das Merkmal, steigt die Vorhersage.

Vorsicht bei der Deutung: Die Merkmale haben unterschiedliche Wertebereiche. `temp` liegt zwischen 0 und 1, `mnth` zwischen 1 und 12. Die Gewichte lassen sich daher nur bedingt direkt miteinander vergleichen.

In [ ]:
gewichte = pd.Series(regression.coef_, index=features).sort_values()
gewichte.plot.barh(figsize=(8, 4), title="Gelernte Gewichte der linearen Regression")
plt.grid()
plt.show()

## 7. Modell testen

Jetzt lassen wir das Modell die Testdaten vorhersagen, also Tage, die es beim Training nicht gesehen hat. Dazu verwenden wir die Methode `predict()`.

**Lücke:** Welche Daten übergeben wir der Methode `predict()`, um Vorhersagen für die Testtage zu erhalten?

In [ ]:
predictions = regression.predict(XXXX)
print(predictions[:10].round())

Das sind die vorhergesagten Ausleihen für die ersten zehn Testtage. Wie gut sind diese Vorhersagen? Am anschaulichsten ist zunächst der Vergleich im Diagramm.

In [ ]:
test_datum = df.iloc[train_size:]["dteday"]

fig, ax = plt.subplots()
ax.plot(test_datum, y_test, label="Tatsächliche Nachfrage")
ax.plot(test_datum, predictions, label="Vorhersage lineare Regression")
ax.set_ylabel("Ausleihen pro Tag")
ax.legend()
ax.grid()
plt.show()

Das Modell hat den Wochenrhythmus und die grobe Saison erkannt. Ehrlich gesagt ist es aber noch nicht besonders präzise: Die Spitzen am Wochenende werden deutlich unterschätzt. Um das in Zahlen zu fassen, nutzen wir **Gütemaße**.

## 8. Modell bewerten

### Mittlerer absoluter Fehler (MAE)
Der **Mean Absolute Error** gibt an, um wie viel die Vorhersage im Durchschnitt danebenliegt, und zwar in der Einheit der Zielgröße, hier also in Fahrrädern. Er ist dadurch besonders leicht zu verstehen:

<p align="center"><b>MAE = 1/N · Σ | y<sub>i</sub> − ŷ<sub>i</sub> |</b></p>

Dabei ist y<sub>i</sub> der tatsächliche Wert am Tag i, ŷ<sub>i</sub> die Vorhersage für diesen Tag und N die Anzahl der Testtage. Das Zeichen Σ steht für die Summe über alle Tage.

### Mittlerer quadratischer Fehler (MSE)
Der **Mean Squared Error** quadriert die Abweichungen, bevor gemittelt wird. Große Fehler fallen dadurch stärker ins Gewicht. Viele Lernverfahren minimieren beim Training genau diese Größe:

<p align="center"><b>MSE = 1/N · Σ ( y<sub>i</sub> − ŷ<sub>i</sub> )²</b></p>

Wir berechnen beide Werte, jeweils für die Testdaten **und** für die Trainingsdaten.

**Lücken:** Setzen Sie in die Ausgaben die passend benannten Variablen ein.

In [ ]:
from sklearn import metrics

# Fehler auf den Testdaten (unbekannte Tage)
mae_test = metrics.mean_absolute_error(y_test, predictions)
mse_test = metrics.mean_squared_error(y_test, predictions)

# Fehler auf den Trainingsdaten (bekannte Tage)
predictions_train = regression.predict(X_train)
mae_train = metrics.mean_absolute_error(y_train, predictions_train)
mse_train = metrics.mean_squared_error(y_train, predictions_train)

print(f"MAE Testdaten:     {XXXX:8.0f} Fahrräder")
print(f"MAE Trainingsdaten: {XXXX:7.0f} Fahrräder")
print(f"MSE Testdaten:     {XXXX:8.0f}")
print(f"MSE Trainingsdaten: {XXXX:7.0f}")

Auf den Trainingsdaten ist der Fehler kleiner, denn diese Tage kennt das Modell bereits. Entscheidend für die Praxis ist immer der Fehler auf den **Testdaten**.

### Bestimmtheitsmaß R²
Das **Bestimmtheitsmaß** R² gibt an, welcher Anteil der Schwankungen in der Zielgröße durch das Modell erklärt wird:

<p align="center"><b>R² = 1 − Σ ( y<sub>i</sub> − ŷ<sub>i</sub> )² / Σ ( y<sub>i</sub> − ȳ )²</b></p>

Im Zähler steht der quadrierte Fehler des Modells, im Nenner der quadrierte Fehler, den man macht, wenn man immer nur den Mittelwert vorhersagt.

Dabei ist ȳ der Mittelwert der tatsächlichen Werte.
- R² = 1: perfekte Vorhersage
- R² = 0: nicht besser, als immer den Mittelwert zu raten
- R² < 0: sogar schlechter als der Mittelwert

In `scikit-learn` liefert die Methode `score()` direkt das R².

**Lücken:** Setzen Sie die passenden Variablen ein.

In [ ]:
r2_test = regression.score(X_test, y_test)
r2_train = regression.score(X_train, y_train)

print(f"R² Testdaten:      {XXXX:.3f}")
print(f"R² Trainingsdaten: {XXXX:.3f}")

### Ist das jetzt gut? Der Vergleich mit einer einfachen Faustregel
Ein Gütemaß allein sagt wenig. Man sollte ein Modell immer mit einer **naiven Faustregel** vergleichen, der sogenannten *Baseline*. Die einfachste Faustregel lautet: „Ich sage jeden Tag den Durchschnitt der Trainingsdaten vorher.“ Ein Modell lohnt sich nur, wenn es diese Faustregel deutlich schlägt.

In [ ]:
baseline = np.full(len(y_test), y_train.mean())
mae_baseline = metrics.mean_absolute_error(y_test, baseline)

print(f"MAE Faustregel (immer Mittelwert): {mae_baseline:.0f} Fahrräder")
print(f"MAE lineare Regression:            {mae_test:.0f} Fahrräder")

Die lineare Regression ist deutlich besser als die Faustregel. Es bleibt aber noch viel Luft nach oben. Wir sammeln die Ergebnisse, um später alle Modelle vergleichen zu können.

In [ ]:
ergebnisse = {}
ergebnisse["Faustregel (Mittelwert)"] = {"MAE Test": mae_baseline, "R² Test": metrics.r2_score(y_test, baseline)}
ergebnisse["Lineare Regression"] = {"MAE Test": mae_test, "R² Test": r2_test}
vorhersagen = {"Lineare Regression": predictions}

## 9. Ein stärkeres Modell: Random Forest

Die lineare Regression kann nur „gerade“ Zusammenhänge abbilden. In unseren Daten wirken die Merkmale aber zusammen. Ein warmer Tag erhöht die Nachfrage vor allem dann stark, wenn er auf ein Wochenende fällt.

Ein **Random Forest** kommt mit solchen Zusammenhängen gut zurecht. Er besteht aus vielen **Entscheidungsbäumen**, von denen jeder eine Reihe einfacher Ja/Nein-Fragen stellt, etwa „Ist `workingday` = 0?“ oder „Ist `temp` > 0,5?“. Jeder Baum lernt auf einer etwas anderen Stichprobe der Trainingsdaten. Die Vorhersage ist der Mittelwert aller Bäume. Random Forests sind in der Industrie für tabellarische Daten sehr beliebt, weil sie robust sind und wenig Feintuning brauchen.

Das Schöne an `scikit-learn`: Die Bedienung ist exakt dieselbe wie eben.

**Lücken:**
- Legen Sie die Anzahl der Bäume fest (`n_estimators`). 100 ist ein guter Startwert.
- Trainieren Sie das Modell mit der richtigen Methode.

`random_state=0` sorgt dafür, dass bei jedem Durchlauf dasselbe Ergebnis herauskommt. Der Zufall ist damit reproduzierbar.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=XXXX, random_state=0)
forest.XXXX(X_train, y_train)

predictions_forest = forest.predict(X_test)
mae_forest = metrics.mean_absolute_error(y_test, predictions_forest)

print(f"MAE Testdaten:      {mae_forest:.0f} Fahrräder")
print(f"R² Testdaten:       {forest.score(X_test, y_test):.3f}")
print(f"R² Trainingsdaten:  {forest.score(X_train, y_train):.3f}")

Der Fehler auf den Testdaten ist deutlich kleiner geworden.

Auffällig ist aber der große Abstand zwischen Training (R² nahe 1) und Test. Der Random Forest hat sich die Trainingsdaten teilweise **auswendig gemerkt**. Man spricht von **Überanpassung** (engl. *Overfitting*). Solange das Ergebnis auf den Testdaten gut ist, ist das nicht schlimm. Es zeigt aber, warum man ein Modell **nie** nur auf den Trainingsdaten bewerten darf.

Ein weiterer Vorteil: Ein Random Forest verrät, welche Merkmale ihm am wichtigsten waren, die sogenannte *Feature Importance*.

In [ ]:
wichtigkeit = pd.Series(forest.feature_importances_, index=features).sort_values()
wichtigkeit.plot.barh(figsize=(8, 4), title="Wichtigkeit der Merkmale im Random Forest")
plt.grid()
plt.show()

Passt das zu Ihrer Intuition? Temperatur und Arbeitstag ja oder nein bestimmen am stärksten, wie viele Gelegenheitsnutzer unterwegs sind. Genau solche Erkenntnisse sind in der Praxis oft genauso wertvoll wie die Vorhersage selbst, etwa die Frage, welche Prozessparameter die Qualität am stärksten beeinflussen.

In [ ]:
ergebnisse["Random Forest"] = {"MAE Test": mae_forest, "R² Test": forest.score(X_test, y_test)}
vorhersagen["Random Forest"] = predictions_forest

## 10. Neuronales Netz

Der Hype um Künstliche Intelligenz entstand vor allem durch **Deep Learning**, also durch neuronale Netze mit mehreren Schichten. Auch `scikit-learn` enthält ein einfaches neuronales Netz, den **`MLPRegressor`** (*Multi-Layer Perceptron*).

Ein neuronales Netz besteht aus Schichten von *Neuronen*. Jedes Neuron bildet eine gewichtete Summe seiner Eingänge und wendet darauf eine einfache nichtlineare Funktion an. Beim Training werden die Gewichte in vielen Durchläufen über die Trainingsdaten (*Epochen*) schrittweise so angepasst, dass der Fehler sinkt.

Neuronale Netze sind sehr flexibel, stellen aber höhere Ansprüche an die Aufbereitung der Daten und an die Einstellungen. Beides sehen wir uns in diesem Abschnitt an.

### Schritt 1: Merkmale passend aufbereiten (One-Hot-Kodierung)

In Abschnitt 6 haben wir die Wochentage einfach durchnummeriert: Montag = 0 bis Sonntag = 6. Für ein neuronales Netz ist das irreführend. Es rechnet mit diesen Zahlen und „glaubt“ dadurch, der Sonntag (6) sei sechsmal so viel wie der Dienstag (1) und weit weg vom Montag (0). Mit dem Wochenrhythmus hat das nichts zu tun.

Besser ist die **One-Hot-Kodierung**: Aus der Spalte `weekday` werden sieben Spalten `weekday_0` bis `weekday_6`. In jeder Zeile steht genau in der Spalte des passenden Wochentags eine 1, in allen anderen eine 0. So kann das Netz für jeden Wochentag ein eigenes Gewicht lernen. Dasselbe machen wir mit den anderen Kategorie-Spalten.

Die Funktion `pd.get_dummies()` übernimmt das für uns.

**Lücken:** Ergänzen Sie die Spalten für den Wochentag und die Jahreszeit.

In [ ]:
kategorien = ["XXXX", "XXXX", "weathersit", "mnth"]

X_nn = pd.get_dummies(df[features], columns=kategorien, dtype=float)
X_train_nn = X_nn.iloc[:train_size]
X_test_nn = X_nn.iloc[train_size:]

print("Anzahl Merkmale vorher:", len(features), "| nachher:", X_nn.shape[1])
X_nn.head()

### Schritt 2: Validierungsdaten und Lernkurve

Beim neuronalen Netz wollen wir **während** des Trainings beobachten, ob es wirklich etwas Allgemeingültiges lernt oder sich nur die Trainingsdaten merkt. Die Testdaten dürfen wir dafür nicht anfassen. Sie bleiben bis ganz zum Schluss unter Verschluss, sonst wäre der Test nicht mehr ehrlich.

Deshalb zweigen wir von den Trainingsdaten noch einmal 20 % als **Validierungsdaten** ab. Damit haben wir drei Datensätze:

| Datensatz | Wofür? |
|---|---|
| Trainingsdaten | Daraus lernt das Netz die Gewichte. |
| Validierungsdaten | Damit prüfen wir nach jeder Epoche, wie gut das Netz auf unbekannten Daten ist. |
| Testdaten | Damit bewerten wir das fertige Modell ein einziges Mal am Ende. |

Nach jeder Epoche berechnen wir den Fehler (MSE) auf den Trainings- **und** auf den Validierungsdaten. Trägt man beide über die Epochen auf, erhält man die **Lernkurve**. Zusätzlich merken wir uns das Netz aus der Epoche mit dem kleinsten Validierungsfehler. Dieses Vorgehen heißt **Early Stopping**: Wir nehmen das Netz zu dem Zeitpunkt, an dem es auf unbekannten Daten am besten war.

Die folgende Funktion erledigt das. Sie müssen sie nicht im Detail verstehen, die Kommentare beschreiben jeden Schritt.

**Lücke:** Die Methode `partial_fit()` trainiert das Netz genau eine Epoche lang. Ergänzen Sie den Methodennamen in der Trainingsschleife.

In [ ]:
import copy
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")


def trainiere_netz(X, y, neuronen, schichten, regularisierung, lernrate, epochen):
    # 1. 20 % der Trainingsdaten zufällig als Validierungsdaten abzweigen
    X_fit, X_val, y_fit, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

    # 2. Merkmale skalieren (Mittelwert 0, Standardabweichung 1) – wichtig für neuronale Netze
    scaler = StandardScaler().fit(X_fit)
    X_fit_s = scaler.transform(X_fit)
    X_val_s = scaler.transform(X_val)

    # 3. Das neuronale Netz anlegen
    netz = MLPRegressor(hidden_layer_sizes=(neuronen,) * schichten, alpha=regularisierung,
                        learning_rate_init=lernrate, batch_size=32, random_state=0)

    # 4. Epoche für Epoche trainieren und die Fehler mitschreiben
    verlauf = {"Training": [], "Validierung": []}
    bester_fehler, bestes_netz, beste_epoche = float("inf"), None, 0
    for epoche in range(epochen):
        netz.XXXX(X_fit_s, y_fit)
        fehler_train = metrics.mean_squared_error(y_fit, netz.predict(X_fit_s))
        fehler_val = metrics.mean_squared_error(y_val, netz.predict(X_val_s))
        verlauf["Training"].append(fehler_train)
        verlauf["Validierung"].append(fehler_val)
        # Early Stopping: Netz mit dem kleinsten Validierungsfehler merken
        if fehler_val < bester_fehler:
            bester_fehler, bestes_netz, beste_epoche = fehler_val, copy.deepcopy(netz), epoche

    # 5. Skalierung und bestes Netz zu einem fertigen Modell zusammenfügen
    modell = make_pipeline(scaler, bestes_netz)
    return modell, verlauf, beste_epoche


def zeige_lernkurve(verlauf, beste_epoche, titel):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(verlauf["Training"], label="Fehler auf Trainingsdaten")
    ax.plot(verlauf["Validierung"], label="Fehler auf Validierungsdaten")
    ax.axvline(beste_epoche, color="grey", linestyle="--", label=f"Beste Epoche ({beste_epoche})")
    ax.set_xlabel("Epoche")
    ax.set_ylabel("Fehler (MSE)")
    ax.set_yscale("log")
    ax.set_title(titel)
    ax.legend()
    ax.grid()
    plt.show()

print("Funktionen bereit.")

### Schritt 3: Hyperparameter festlegen

Anders als die Gewichte werden **Hyperparameter** nicht gelernt, sondern vorab von uns festgelegt. Gute Werte findet man durch Ausprobieren. Die folgenden Werte haben sich für diesen Datensatz bewährt:

In [ ]:
# Anzahl der Neuronen pro verdeckter Schicht
neuronen_pro_schicht = 32
# Anzahl der verdeckten Schichten
anzahl_schichten = 2
# Regularisierung: bestraft zu große Gewichte und wirkt so dem Auswendiglernen entgegen
regularisierung = 10.0
# Lernrate: wie große Schritte das Netz bei jeder Anpassung der Gewichte macht
lernrate = 0.003
# Anzahl der Durchläufe über die Trainingsdaten
epochen = 300

### Schritt 4: Netz trainieren

Das Training dauert im Browser etwa 10 bis 30 Sekunden. Solange links neben der Zelle `[*]` steht, wird noch gerechnet.

**Lücken:** Übergeben Sie die vorbereiteten Merkmale für das neuronale Netz (aus Schritt 1) sowie die Hyperparameter aus Schritt 3.

In [ ]:
netz, verlauf, beste_epoche = trainiere_netz(XXXX, y_train,
                                             neuronen=XXXX,
                                             schichten=XXXX,
                                             regularisierung=XXXX,
                                             lernrate=XXXX,
                                             epochen=XXXX)

predictions_netz = netz.predict(X_test_nn)
mae_netz = metrics.mean_absolute_error(y_test, predictions_netz)

print(f"Beste Epoche:      {beste_epoche}")
print(f"MAE Testdaten:     {mae_netz:.0f} Fahrräder")
print(f"R² Testdaten:      {netz.score(X_test_nn, y_test):.3f}")

### Schritt 5: Die Lernkurve lesen

In [ ]:
zeige_lernkurve(verlauf, beste_epoche, "Lernkurve des neuronalen Netzes")

So lesen Sie das Diagramm:
- Am Anfang fallen **beide** Kurven stark. Das Netz lernt die groben Muster, die für alle Tage gelten.
- Der Trainingsfehler liegt immer unter dem Validierungsfehler, denn die Trainingstage kennt das Netz.
- Solange auch der **Validierungsfehler** noch sinkt, lernt das Netz etwas Allgemeingültiges.
- Irgendwann sinkt nur noch der Trainingsfehler, während der Validierungsfehler stagniert oder leicht steigt. Ab hier beginnt das Netz, Einzelheiten der Trainingstage auswendig zu lernen. Die gestrichelte Linie zeigt die Epoche, deren Netz wir dank Early Stopping verwenden. Die Regularisierung sorgt dafür, dass der Validierungsfehler danach nur langsam ansteigt.

In [ ]:
ergebnisse["Neuronales Netz"] = {"MAE Test": mae_netz, "R² Test": netz.score(X_test_nn, y_test)}
vorhersagen["Neuronales Netz"] = predictions_netz

### Schritt 6: Überanpassung live erleben

Was passiert, wenn wir das Netz deutlich größer machen und die Regularisierung fast abschalten? Ein solches Netz hat genug „Speicherplatz“, um sich die Trainingsdaten auswendig zu merken.

Das Training dieses größeren Netzes dauert etwas länger.

In [ ]:
netz_gross, verlauf_gross, beste_epoche_gross = trainiere_netz(X_train_nn, y_train,
                                                               neuronen=128, schichten=3,
                                                               regularisierung=0.0001,
                                                               lernrate=0.003, epochen=250)
zeige_lernkurve(verlauf_gross, beste_epoche_gross, "Großes Netz ohne Regularisierung")

Jetzt sieht man **Überanpassung** (*Overfitting*) ganz deutlich:
- Der Trainingsfehler sinkt immer weiter. Das Netz merkt sich die Trainingstage immer genauer.
- Der Validierungsfehler erreicht dagegen schon früh sein Minimum und **steigt danach wieder an**. Ab diesem Punkt lernt das Netz nur noch Zufälligkeiten der Trainingsdaten auswendig, die auf neuen Tagen nicht gelten.
- Die Schere zwischen beiden Kurven ist das typische Erkennungszeichen von Überanpassung.
- Die Kurven sind außerdem unruhig, mit einzelnen Ausschlägen. Ein großes Netz ohne Regularisierung reagiert empfindlich auf jede Anpassung der Gewichte.

Hätten wir das Netz aus der letzten Epoche genommen, wäre es schlechter, als es sein müsste. Early Stopping schützt davor, indem es das Netz aus der besten Epoche verwendet. Noch besser ist es, die Überanpassung durch eine passende Größe und Regularisierung von vornherein zu bremsen, wie in Schritt 4.

### Ihre Aufgabe: Hyperparameter ausprobieren
Gehen Sie zurück zu Schritt 3, ändern Sie die Werte und führen Sie danach die Zellen von Schritt 4 und 5 erneut aus. Beobachten Sie den MAE und die Lernkurve. Ideen:
- Weniger Neuronen, z. B. `8`, oder mehr, z. B. `128`. Mehr Neuronen bedeuten auch eine längere Rechenzeit.
- Nur eine Schicht oder drei Schichten
- Schwächere (`0.0001`) oder stärkere (`50.0`) Regularisierung
- Eine größere Lernrate (`0.03`): Wird die Lernkurve unruhiger?
- Weniger Epochen, z. B. `30`. Ist das Netz dann schon fertig?

Wer findet das beste Netz? Und schlägt es den Random Forest?

## 11. Alle Modelle im Vergleich

Zum Schluss stellen wir alle Modelle nebeneinander, zuerst als Tabelle und dann im Diagramm.

In [ ]:
vergleich = pd.DataFrame(ergebnisse).T.sort_values("MAE Test")
vergleich.round({"MAE Test": 0, "R² Test": 3})

In [ ]:
fig, ax = plt.subplots()
ax.plot(test_datum, y_test, color="black", linewidth=2, label="Tatsächliche Nachfrage")
for name, werte in vorhersagen.items():
    ax.plot(test_datum, werte, label=name, alpha=0.8)
ax.set_ylabel("Ausleihen pro Tag")
ax.legend()
ax.grid()
plt.show()

**Einige Beobachtungen zum Mitnehmen:**
- Auch ein einfaches Modell kann schon einen großen Teil der Muster erfassen. Man sollte immer einfach anfangen.
- Ein komplexeres Modell ist nicht automatisch besser. Das neuronale Netz kommt nah an den Random Forest heran, aber nur mit passender Datenaufbereitung, Validierungsdaten und abgestimmten Hyperparametern. Bei kleinen, tabellarischen Datensätzen wie diesem (rund 600 Trainingstage) sind Random Forests oft mindestens so gut, lassen sich leichter erklären und brauchen kaum Feintuning. Neuronale Netze spielen ihre Stärken vor allem bei großen Datenmengen und bei Bildern, Sprache oder Zeitreihen aus.
- Lernkurven mit Trainings- **und** Validierungsfehler zeigen, ob ein Modell auswendig lernt.
- Die meiste Arbeit steckt in den **Daten**: Ausreißer erkennen, fehlende Werte behandeln, Text in Zahlen umwandeln und Merkmale skalieren.

## 12. Modell speichern und laden

Damit man ein trainiertes Modell später wieder verwenden kann, etwa in einer Anwendung an der Maschine, speichert man es als Datei. Dafür nutzen wir das in Python eingebaute Modul `pickle`.

Wir speichern hier den Random Forest. Die Datei erscheint danach links im Dateibrowser, gegebenenfalls nach einem Klick auf das Aktualisieren-Symbol.

In [ ]:
import pickle

dateiname = "fahrrad_modell.pkl"
with open(dateiname, "wb") as datei:
    pickle.dump(forest, datei)
print("Modell gespeichert als", dateiname)

Genauso einfach lässt sich das Modell wieder laden. Zur Kontrolle berechnen wir das R² auf den Testdaten. Es muss exakt derselbe Wert herauskommen wie vorher.

In [ ]:
with open(dateiname, "rb") as datei:
    geladenes_modell = pickle.load(datei)

print(f"R² des geladenen Modells: {geladenes_modell.score(X_test, y_test):.3f}")

## 13. Zum Weiterprobieren

Wenn Sie noch Zeit haben, probieren Sie eine oder mehrere dieser Aufgaben aus:

1. **Andere Zielgröße:** Ändern Sie in Abschnitt 3 die Zielgröße auf `registered` oder `cnt` und führen Sie danach alle Zellen erneut aus (*Kernel → Restart Kernel and Run All Cells…*). Achtung: Die Bereinigung in Abschnitt 4 filtert die Spalte `casual`. Überlegen Sie, ob Sie diesen Schritt anpassen müssen, und schauen Sie sich das Diagramm der neuen Zielgröße an. Welche Merkmale werden jetzt wichtig? Registrierte Nutzer fahren oft zur Arbeit.
2. **Weniger Merkmale:** Entfernen Sie einzelne Merkmale aus der Liste `features`, zum Beispiel `atemp`, das fast dasselbe aussagt wie `temp`. Wird das Modell schlechter?
3. **Zufällige statt zeitlicher Aufteilung:** Was passiert mit den Ergebnissen, wenn man die Tage zufällig in Training und Test aufteilt? Tipp: `from sklearn.model_selection import train_test_split`. Warum ist das Ergebnis dann vermutlich zu optimistisch?
4. **Bereinigung kritisch prüfen:** In `describe()` hatte die Luftfeuchtigkeit `hum` einen Minimalwert von 0. Ist das plausibel? Wie würden Sie damit umgehen?